# Fase 4 — Fine-tuning do PTT5 (Encoder-Decoder)

O **PTT5** (`unicamp-dl/ptt5-base-portuguese-vocab`) aprende a traduzir o resultado estruturado
do exame para linguagem simples.

**Entrada** (já classificada pelas regras):
`triagem: CRITICO | Hemoglobina: 6,4 g/dL (ref 12 a 17,5 g/dL; baixo) | ...`

**Saída:** explicação para o paciente. Saudação, orientação de conduta (ex.: ligar 192) e aviso legal
são fixos na API — o modelo nunca decide a conduta nem o nível de risco.

Métricas: **ROUGE-L**, **Flesch PT-BR** (laudo bruto × texto gerado), **fidelidade numérica**
(nenhum número inventado) e **violações de guardrail**.

> No Colab: *Ambiente de execução → Alterar o tipo → GPU T4* (~10–20 min). Na CPU leva horas.

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    # Repositório privado? Use https://<usuario>:<token>@gitlab.com/... ou envie um .zip do projeto.
    if not os.path.exists("caresync-nlp-core"):
        !git clone -q https://gitlab.com/caiofasaraiva/caresync-nlp-core.git
    %cd caresync-nlp-core
    !pip install -q "transformers>=5" datasets accelerate sentencepiece protobuf seqeval pydantic-settings groq fastapi
else:
    %cd ..
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (CPU)")

In [ ]:
!python scripts/gerar_dataset.py --n 1200 --seed 42

## Treinamento

In [ ]:
!python scripts/treinar_ptt5.py --epocas 8 --lr 3e-4 --batch 8

## Curva de loss

In [ ]:
ARQUIVO = "resultados/ptt5_ptt5-caresync.json"
TITULO = "PTT5 — fine-tuning"

import json
import matplotlib.pyplot as plt

res = json.load(open(ARQUIVO, encoding="utf-8"))
hist = res["historico"]
treino = [(h["epoch"], h["loss"]) for h in hist if "loss" in h]
valid = [(h["epoch"], h["eval_loss"]) for h in hist if "eval_loss" in h]

plt.figure(figsize=(7, 4))
plt.plot(*zip(*treino), label="treino")
plt.plot(*zip(*valid), "o-", label="validação")
plt.xlabel("época"); plt.ylabel("loss"); plt.title(TITULO); plt.legend(); plt.grid(alpha=.3)
plt.savefig(ARQUIVO.replace(".json", "_loss.png"), dpi=200, bbox_inches="tight")
plt.show()

## Métricas no conjunto de teste

In [ ]:
import pandas as pd

pd.Series(res["teste"]).round(3).to_frame("valor")

## Legibilidade: laudo bruto × texto gerado (Flesch PT-BR)
75–100 muito fácil · 50–75 fácil · 25–50 difícil · 0–25 muito difícil

In [ ]:
df = pd.DataFrame(res["amostras"])
plt.figure(figsize=(7, 4))
plt.hist(df["flesch_laudo"], bins=20, alpha=.6, label="laudo bruto")
plt.hist(df["flesch_gerado"], bins=20, alpha=.6, label="texto gerado (PTT5)")
plt.axvline(50, color="gray", ls="--", lw=1)
plt.xlabel("índice de Flesch (PT-BR)"); plt.ylabel("laudos"); plt.legend(); plt.grid(alpha=.3)
plt.savefig("resultados/ptt5_flesch.png", dpi=200, bbox_inches="tight"); plt.show()
df.groupby("risco")[["rouge_l", "flesch_gerado", "fiel_numeros", "viola_guardrail"]].mean().round(3)

## Exemplos gerados

In [ ]:
for _, a in df.groupby("risco").head(1).iterrows():
    print(f"[{a.risco}] ENTRADA: {a.entrada}\n\nGERADO:\n{a.gerado}\n\nREFERÊNCIA:\n{a.referencia}\n" + "-" * 80)

## Baixar o modelo treinado
Coloque a pasta em `modelos/ptt5-caresync` e use `GERADOR=ptt5` na API.

In [ ]:
if "google.colab" in sys.modules:
    !zip -qr ptt5-caresync.zip modelos/ptt5-caresync -x "modelos/ptt5-caresync/checkpoints/*"
    from google.colab import files
    files.download("ptt5-caresync.zip")